#### Inspect top-level Kaggle input directory to locate the competition dataset

In [1]:
import os

print("Input:", os.listdir("/kaggle/input"))
comp_dir = "/kaggle/input/state-farm-distracted-driver-detection"
if os.path.exists(comp_dir):
  print("Inside competition:", os.listdir(comp_dir))
  imgs_dir = os.path.join(comp_dir, "imgs")
  if os.path.exists(imgs_dir):
    print("Inside imgs:", os.listdir(imgs_dir))

Input: ['competitions']


#### Explore competition directory structure, CSV files, and training class folders (c0-c9)

In [2]:
import os

base_dir = (
    "/kaggle/input/competitions/state-farm-distracted-driver-detection"
)
print("Competition contents:", os.listdir(base_dir))

imgs_dir = os.path.join(base_dir, "imgs")
if os.path.exists(imgs_dir):
  print("Imgs contents:", os.listdir(imgs_dir))
  train_dir = os.path.join(imgs_dir, "train")
  if os.path.exists(train_dir):
    print("Classes:", sorted(os.listdir(train_dir)))

Competition contents: ['sample_submission.csv', 'driver_imgs_list.csv', 'imgs']
Imgs contents: ['test', 'train']
Classes: ['c0', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9']


#### Build a pandas DataFrame of labeled training images and count samples per class

In [3]:
from pathlib import Path
import pandas as pd

train_dir = Path(
    "/kaggle/input/competitions/state-farm-distracted-driver-detection/imgs/train"
)

rows = []

for cls_dir in sorted(train_dir.iterdir()):
  if cls_dir.is_dir():
    for img_path in cls_dir.iterdir():
      if img_path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        rows.append({"filepath": str(img_path), "label": cls_dir.name})

df = pd.DataFrame(rows)

print("Total images:", len(df))
print("\nImages per class:")
print(df["label"].value_counts().sort_index())
print("\nFirst 5 rows:")
print(df.head())


Total images: 22424

Images per class:
label
c0    2489
c1    2267
c2    2317
c3    2346
c4    2326
c5    2312
c6    2325
c7    2002
c8    1911
c9    2129
Name: count, dtype: int64

First 5 rows:
                                            filepath label
0  /kaggle/input/competitions/state-farm-distract...    c0
1  /kaggle/input/competitions/state-farm-distract...    c0
2  /kaggle/input/competitions/state-farm-distract...    c0
3  /kaggle/input/competitions/state-farm-distract...    c0
4  /kaggle/input/competitions/state-farm-distract...    c0


#### Perform a stratified 70/15/15 train/validation/test split before any preprocessing or synthesis

In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Test size:", len(test_df))

Train size: 15696
Val size: 3364
Test size: 3364


#### Save the combined dataset split table with a 'split' column to CSV for reproducibility

In [5]:
import os
import pandas as pd

train_part = train_df.assign(split="train")
val_part = val_df.assign(split="val")
test_part = test_df.assign(split="test")

split_df = pd.concat([train_part, val_part, test_part], ignore_index=True)

os.makedirs("/kaggle/working/splits", exist_ok=True)
out_path = "/kaggle/working/splits/dataset_split.csv"
split_df.to_csv(out_path, index=False)

print("Saved split file to:", out_path)
print("Total rows:", len(split_df))
print(split_df["split"].value_counts())

Saved split file to: /kaggle/working/splits/dataset_split.csv
Total rows: 22424
split
train    15696
val       3364
test      3364
Name: count, dtype: int64
